<a href="https://colab.research.google.com/github/whrizkey/DepokPropertyPrices/blob/main/notebooks/001_Depok_Property_Prices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import re

file_path = '/content/Raw-Final-Rumah.csv'
df = pd.read_csv(file_path)
# Renaming
column_mapping = {
    'Harga': 'price',
    'Kamar Tidur': 'bedrooms',
    'Kamar Mandi': 'bathrooms',
    'Garasi': 'garage',
    'Luas Tanah': 'land_area_sqm',
    'Luas Bangunan': 'building_area_sqm',
    'Kecamatan': 'district',
    'Page': 'page'
}

df.rename(columns=column_mapping, inplace=True)


## Cleaning the price

def clean_price(text):
  if pd.isna(text):
      return np.nan

  text = str(text).lower()

  text = text.replace('rp', '').replace('total', '').strip()

  multiplier = 1
  if 'juta' in text:
    multiplier = 1_000_000
    text = text.replace('juta', '').strip()
  elif 'miliar' in text:
    multiplier = 1_000_000_000
    text = text.replace('miliar', '').strip()

  text = text.replace(',', '.')

  clean_number_string = re.sub(r'[^\d.]', '', text)

  try:
      return float(clean_number_string) * multiplier
  except ValueError:
      return np.nan

df['price_clean'] = df['price'].apply(clean_price)

pd.options.display.float_format = '{:,.0f}'.format

## Cleaning rooms and area columns

df['bedrooms_clean'] = df['bedrooms'].astype(str).str.extract(r'(\d+)').astype(float)
df['bathrooms_clean'] = df['bathrooms'].astype(str).str.extract(r'(\d+)').astype(float)
df['garage_clean'] = df['garage'].astype(str).str.extract(r'(\d+)').astype(float)


df['land_area_clean'] = df['land_area_sqm'].astype(str).str.extract(r'(\d+)').astype(float)
df['building_area_clean'] = df['building_area_sqm'].astype(str).str.extract(r'(\d+)').astype(float)

df['all_features'] = df['bedrooms'].astype(str) + " | " + \
                     df['bathrooms'].astype(str) + " | " + \
                     df['garage'].astype(str) + " | " + \
                     df['land_area_sqm'].astype(str) + " | " + \
                     df['building_area_sqm'].astype(str)

df['land_area_clean'] = df['all_features'].str.extract(r'LT\s*[:]?\s*(\d+)', flags=re.IGNORECASE).astype(float)
df['building_area_clean'] = df['all_features'].str.extract(r'LB\s*[:]?\s*(\d+)', flags=re.IGNORECASE).astype(float)

def clean_room(text):
    text = str(text).upper()
    if 'LT' in text or 'LB' in text or text == 'NAN':
        return np.nan
    numbers = re.findall(r'\d+', text)
    if numbers:
        return float(numbers[0])
    return np.nan

df['bedrooms_clean'] = df['bedrooms'].apply(clean_room)
df['bathrooms_clean'] = df['bathrooms'].apply(clean_room)
df['garage_clean'] = df['garage'].apply(clean_room)

# Filter room outliers
df.loc[df['bedrooms_clean'] > 20, 'bedrooms_clean'] = np.nan
df.loc[df['bathrooms_clean'] > 20, 'bathrooms_clean'] = np.nan
df.loc[df['garage_clean'] > 20, 'garage_clean'] = np.nan


columns_to_drop = ['bedrooms', 'bathrooms', 'garage', 'land_area_sqm', 'building_area_sqm', 'all_features', 'price']
df_clean = df.drop(columns=columns_to_drop)


df_clean = df_clean.dropna(subset=['price_clean', 'land_area_clean', 'building_area_clean', 'district']).copy()
df_clean = df_clean[(df_clean['price_clean'] > 0) &
                    (df_clean['land_area_clean'] > 0) &
                    (df_clean['building_area_clean'] > 0)]


max_residential_price = 15_000_000_000
Q1 = df_clean['price_clean'].quantile(0.25)
Q3 = df_clean['price_clean'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = min(Q3 + 1.5 * IQR, max_residential_price)
lower_bound = max(Q1 - 1.5 * IQR, 0)

# Final clean dataframe ready for export
df_final = df_clean[(df_clean['price_clean'] >= lower_bound) &
                    (df_clean['price_clean'] <= upper_bound)].copy()


output_file_name = 'Cleaned-Rumah-Depok.csv'


df_final.to_csv(output_file_name, index=False)





✅ Success! Your file has been saved as 'Cleaned-Rumah-Depok.csv'
